In [3]:
import xml.etree.ElementTree as ET
import numpy as np

In [2]:
! mv dilated/dilated_CA.xml dilated_CA.xml.old
! mv constricted/constricted_CA.xml constricted_CA.xml.old

In [6]:
def read_angle(topfile):
    # Read large angle indices
    with open (topfile, "r") as fin:
        index = []
        process_on = False
        for line in fin:
            if line.startswith("[ angles ]"):
                process_on = True
            elif process_on and line.startswith("["):
                process_on = False
            elif process_on:
                try:
                    line_splited = line.split()
                    int(line_splited[0])
                    if float(line_splited[4]) > 150 or float(line_splited[4]) < 30:
                        index.append(list(map(int, line_splited[:3])))
                except:
                    pass
    return np.array(index)

In [55]:
# In CA model, delete the dihedrals where the angle is larger than 150 or smaller than 30
def deleteDihedral(topfile, xmlfile, newfile):
    index = read_angle(topfile)
    tree = ET.parse(xmlfile)
    root = tree.getroot()

    for dihedrals_xml in root.find('dihedrals'):
        for atoms_ijkl in dihedrals_xml.findall('interaction'):
            d = atoms_ijkl.attrib
            if int(d['i']) in index[:, 0] or int(d['j']) in index[:, 0]:
                dihedrals_xml.remove(atoms_ijkl)
                
    tree.write(newfile)


In [56]:
deleteDihedral("dilated/dilated_CA.top", "dilated_CA.xml.old", "dilated/dilated_CA.xml")

In [57]:
deleteDihedral("constricted/constricted_CA.top", "constricted_CA.xml.old", "constricted/constricted_CA.xml")

In [58]:
! echo >> dilated/dilated_CA.xml
! echo >> constricted/constricted_CA.xml